# <p style="text-align: center;">Housing price prediction</p>
Objective: Predict the house price considering the metrics such as Area, bedrooms, bathrooms, stories and other metrics

Author: Karteek Pradyumna Bulusu

In [ ]:
import pandas
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn import metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

Import the dataset from Kaggle

In [ ]:
def read_input():
    df = pandas.read_csv('/kaggle/input/housing-price-prediction/Housing.csv')
    print(f"Shape of the data: {df.shape}")
    print(df.head())
    
    # return Housing data in pandas DataFrame
    return df

In [ ]:
housing_df = read_input()

# Exploratory Data Analysis
Function - eda_metrics(df)
return pandas DataFrame

<b>Description</b>: Perform basic data quality checks and describing the dataframe

In [ ]:
def eda_metrics(df):
    # check if any of the fields have null values
    print(f'Boolean value showing if a field has null values\n\n {df.isnull().any()}')
    # Returns description of the dataframe- count, mean, standard deviation, min and max values of each field in the dataframe.
    print('\n----\nDescription of the dataframe\n')
    
    # return basic statistical details like percentile, mean, std, etc
    return df.describe()
    

In [ ]:
eda_metrics(housing_df)

#### Re-arranging fields

In [ ]:
housing_df['house_id'] = housing_df.index + 1
print(housing_df.columns)

In [ ]:
housing_df = housing_df[[ 'house_id', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad',
       'guestroom', 'basement', 'hotwaterheating', 'airconditioning',
       'parking', 'prefarea', 'furnishingstatus', 'price']]

# Data Cleaning
<br>
Convert all labelled values into categorical values in order to perform correlation analysis and prediction analysis.
Only then those values will have a statistical signifance and can be used as a feature.

Convert labelled values to categorical values
Change:
Yes = 1<br>
No = 0

Create three columns - one for each of the categories part of furnishingstatus column with same name as values.

In [ ]:
def data_cleaning(df):

    df[['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']] = df[['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']].replace({'yes':1, 'no':0})
    # assigning values to each categorical value instead of creating new boolean column for each value to reduce dimension
    # Giving more weight (value = 3) from furnished and 1 to unfurnished
    df['furnishingstatus'] = df['furnishingstatus'].replace({'furnished':3, 'semi-furnished':2, 'unfurnished':1})
    
    # Return cleaned data without any categorical values.
    return df

In [ ]:
housing_df = data_cleaning(housing_df)

### Dimension
12 Features and 1 Target value
Note - Ignorning house_id from feature list since it's just an identifier.
545 rows

In [ ]:
print(housing_df.shape)
housing_df.head()

# Correlation Matrix
Correlation between all the features and the Target

In [ ]:
# Removing house_id from the correlation analysis

def correlation_matrix(correlation_ip,threshold = -1.0):
    corr = correlation_ip.corr()
    ax = sns.heatmap(
        corr, 
        vmin=-1, vmax=1, center=0,
        cmap=sns.diverging_palette(20, 220, n=200),
        square=True
    )
    ax.set_xticklabels(
        ax.get_xticklabels(),
        rotation=45,
        horizontalalignment='right'
    );
    
    correlation_score = correlation_ip.corr(numeric_only=True)
    
    # return correclation matrix of the dataframe
    return correlation_score[correlation_score > threshold]

In [ ]:
correlation_ip = housing_df[['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement',
                         'hotwaterheating', 'airconditioning','parking', 'prefarea', 'furnishingstatus','price']]
correlation_matrix(correlation_ip)

**Correlation Notes**  

There is negative correlation to furnishingstatus which means Price is negatively correlated with unfurnished. That means, the price is more for furnished houses.
Price is highly correlated with Area, bathrooms, stories and bedroom which was expected.
Price of the house is most correlated to area, bathrooms, stories, bedroom in this order.

Removing `hotwaterheating` from features since correlation of that to price is very low = 0.09.

In [ ]:
housing_df_postcorr = housing_df[['house_id','area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement',
                          'airconditioning','parking', 'prefarea', 'furnishingstatus','price']]

---

# Variance Inflation Factor
This is to test the multicollinearity of the features (predictors)

Underdstanding the result:<br>
VIF starts at 1 and has no upper limit<br>
VIF = 1, no correlation between the independent variable and the other variables<br>
VIF exceeding 5 or 10 indicates high multicollinearity between this independent variable and the others<br>

#### reference- https://www.analyticsvidhya.com/blog/2020/03/what-is-multicollinearity/

In [ ]:
# import
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
# X is features list in a pandas dataframe format
# Returns dataframe vif  with VIF score. 
def calc_vif(X):

    # Calculating VIF
    vif = pandas.DataFrame()
    vif["variables"] = X.columns
    vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    
    # Return VIF score for each feature
    return(vif)

In [ ]:
X = housing_df_postcorr[['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement',
                          'airconditioning','parking', 'prefarea', 'furnishingstatus']]
print(calc_vif(X))

### VIF First pass notes:
Bedrooms have very high VIP, but area, bedrooms, bathrooms, stories, mainroad, furnishingstatus come under high VIF.
High multicollinearity occurs between these independent variables and others. 
This means that they can be predicted by other independent variables in the dataset


                       variables    VIF
           0               area  8.373
           1           bedrooms 16.268
           2          bathrooms  9.468
           3            stories  7.882
           4           mainroad  7.144
           5          guestroom  1.467
           6           basement  2.027
           7    airconditioning  1.703
           8            parking  1.937
           9           prefarea  1.482
           10  furnishingstatus  7.394

### Removing Multicollinearity in the data
New field:
area_and_furnishingstatus_and_mainroad_to_bed_and_bath_stories_ratio = (area*furnishingstatus*mainroad)/(bedroom + bathroom + stories)

Remove area, bedrooms, bathrooms, mainroad, furnishingstatus, stories

In [ ]:
# housing_df['bedroom_to_bath_ratio'] = round(housing_df['bedrooms']/housing_df['bathrooms'],3)
housing_df_postcorr['area_and_furnishingstatus_and_mainroad_to_bed_and_bath_stories_ratio'] = round(((housing_df_postcorr['area'] * housing_df_postcorr['furnishingstatus'] * (housing_df['mainroad']+1))/ (housing_df_postcorr['bedrooms']+housing_df_postcorr['bathrooms']+housing_df_postcorr['stories'])),3)
housing_df_postcorr_vif1 = housing_df_postcorr[['house_id', 'area_and_furnishingstatus_and_mainroad_to_bed_and_bath_stories_ratio', 'guestroom', 'basement',
                          'airconditioning','parking', 'prefarea', 'price']]
housing_df_postcorr_vif1.head()

### See if VIF improved

In [ ]:
X = housing_df_postcorr_vif1[['area_and_furnishingstatus_and_mainroad_to_bed_and_bath_stories_ratio', 'guestroom', 'basement',
                          'airconditioning','parking', 'prefarea']]
print(calc_vif(X))

Using VIF, we unified lot of features into one and hence reduced the features list as well. 

we can use this feature list in our regression model

In [ ]:
housing_df_postcorr_vif1.head()

#### Split Features and Target

In [ ]:
# housing_df_postcorr_vif1.columns
# features = housing_df_postcorr_vif1[['area_and_furnishingstatus_and_mainroad_to_bed_and_bath_stories_ratio',
#        'guestroom', 'basement', 'airconditioning', 'parking', 'prefarea']]

# Splitting dataset to Feature and Target
features = housing_df_postcorr_vif1[['area_and_furnishingstatus_and_mainroad_to_bed_and_bath_stories_ratio']] 
features = features.values.reshape(-1, 1)
target = housing_df_postcorr.price
print(f" Shape of features: {features.shape}")
print(f" Shape of target: {target.shape}")

In [ ]:
from sklearn.model_selection import train_test_split

### Train and Test Split
#### Function - train_test_splt(features, target)
return X_train, X_test, y_train, y_test <br>
<b>Description:</b>
Creating function to take the features and target value and split the data into train and test data and return them. <br>Creating this function also increase re-usability and flexibility while experimenting with multiple models

In [ ]:
def train_test_splt(features, target):
    X_train, X_test, y_train, y_test = train_test_split(features,target , 
                                       random_state=104,  
                                       test_size=0.15,
                                       shuffle=True)
    
    # return Train and test data split for Feature and Target in pandas DataFrame format
    return X_train, X_test, y_train, y_test

### Function - Train_test_details(X_train, X_test, y_train, y_test)
print shape of Train and test features and Target <br>
<b>Description:</b>
Creating function to return the shape of all dataframes and to increase re-usability and flexibility while experimenting with multiple models

In [ ]:
def Train_test_details(X_train, X_test, y_train, y_test):
    print("Training feature",X_train.shape)
    print("Test feature",X_test.shape)
    print("Training target",y_train.shape)
    print("Test target",y_test.shape)

### Function - training_model(X_train, X_test, y_train, y_test, model)
returns built model <br>
<b>Description:</b>
Creating function to train the model and provide model score in order to increase re-usability and flexibility while experimenting with multiple models

In [ ]:
def training_model(X_train, X_test, y_train, y_test, model):
    model.fit(X_train, y_train)
    # Use score method to get accuracy of model
    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)
    print(f'Training score : {train_score}')
    print(f'Test score : {test_score}')
    
    # return the built model
    return model

### Function - predictions_phase(df, model)
returns predictions in pandas DataFrame <br>
<b>Description:</b>
Creating function to perform predcitions and return predicted values in order to increase re-usability and flexibility while experimenting with multiple models

In [ ]:
def predictions_phase(df, model):
    predictions = model.predict(df)
    
    # return the predictions in pandas DataFrame
    return predictions

### Function - prediction_dataframes(train_predictions,y_train,test_predictions, y_test)
returns two pandas dataframes: 1 with y_test and y_predictions (test_predictions); second with y_train and y_predictions (train_predictions)

<b>Description</b>: function to currate dataframe to see actual data and predictions side by side. Creating this function increases re-usability and flexibility while experimenting with multiple models

In [ ]:
def prediction_dataframes(train_predictions,y_train,test_predictions, y_test):
    final_test_predictions = [round(x,6) for x in test_predictions]
    final_train_predictions = [round(x,6) for x in train_predictions]
    
    # Dataframe for test predictions
    test_predictions_andy_test = pandas.DataFrame()
    test_predictions_andy_test['y_test'] = y_test
    test_predictions_andy_test['test_predictions'] = final_test_predictions

    # Dataframe for train predictions
    train_predictions_andy_train = pandas.DataFrame()
    train_predictions_andy_train['y_train'] = y_train
    train_predictions_andy_train['train_predictions'] = final_train_predictions
    
    # return the values in pandas DataFrame
    return test_predictions_andy_test, train_predictions_andy_train

### Function - error_analysis(target, predictions)
returns MAE, MSE and r2 score

<b>Description</b>: function to perform error analysis based on the actual data and predictions. Creating this function increases re-usability and flexibility while experimenting with multiple models

In [ ]:
def error_analysis(target, predictions):
    mae = mean_absolute_error(target, predictions)
    mse = mean_squared_error(target, predictions)
    r2 = r2_score(target, predictions)
    
    # Return error metrics
    return f'MAE : {mae};\t MSE : {mse};\t r2 : {r2}'

---

# LinearRegression on feature prepared from Variance inflation Factor process
Predicting using Linear Regression model
The data considered is feature identified from Variance inflation Factor process

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, target)

In [ ]:
Train_test_details(X_train, X_test, y_train, y_test)

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
model = LinearRegression()
LinearRegr = training_model(X_train, X_test, y_train, y_test, model)

In [ ]:
train_predictions = predictions_phase(X_train, LinearRegr)
test_predictions = predictions_phase(X_test, LinearRegr)

In [ ]:
pandas.set_option('display.float_format', lambda x: '%.3f' % x)

In [ ]:
test_predictions_andy_test, train_predictions_andy_train = prediction_dataframes(train_predictions,y_train,test_predictions, y_test)

In [ ]:
print(f'Train data prediction score \n {error_analysis(y_train, train_predictions)}')
print(f'Test data prediction score \n {error_analysis(y_test, test_predictions)}')

#### plot train and test predictions

In [ ]:
def plot_predictions(X_train, y_train, train_predictions, X_test, y_test, test_predictions):
    plt.figure(figsize=(16, 8))

    # train data
    plt.subplot(121)
    plt.scatter(X_train, y_train)
    plt.yscale('log')
    plt.title("Train data")
    # for i, degree in enumerate(degrees):    
    plt.scatter(X_train, train_predictions, s=15)
    plt.legend(loc='upper left')

    # test data
    plt.subplot(122)
    plt.scatter(X_test, y_test)
    plt.yscale('log')
    plt.title("Test data")
    # for i, degree in enumerate(degrees):    
    plt.scatter(X_test, test_predictions)
    plt.legend(loc='upper left')

In [ ]:
plot_predictions(X_train, y_train, train_predictions, X_test, y_test, test_predictions)

### Conclusion Notes on Variance inflation Factor and LinearRegression
r2 of Training data : 0.11837052139603921<br>
r2 of Test data : 0.05242703684129213
<br>
Tried Variance Inflation Factor and unified multiple fields to reduce the correlation between the fields. The problem is identifying the appropriate way to unify multiple fields to capture the importance of multiple fields into one and drop those individual fields. 
<br>The predictions are very weak and almost neglible. This concludes that it is tough and was incorrect way to combine the features together in order to reduce the multicollinearity.

---

# MinMaxScaler
Hypothesis - Scaling down the features and target to [0.1] range will help with bias variance trade off and result in better model fit and predictions<br>
Transform the dataframe to [0,1] range

In [ ]:
from sklearn.preprocessing import MinMaxScaler

### Function - minmax_scale(df)
return scaled dataframe in pandas DataFrame format<br>

<b>Description</b>: Scaling the dataset in [0,1] range and return the scaled dataframe.

In [ ]:
def minmax_scale(df):
    df_columns = df.columns
    scaler = MinMaxScaler()
    df = scaler.fit_transform(df)

    df = pandas.DataFrame(df)
    df.columns = df_columns
    
    # return scaled data in pandas DataFrame
    return df

In [ ]:
df1 = housing_df_postcorr_vif1[['area_and_furnishingstatus_and_mainroad_to_bed_and_bath_stories_ratio', 'price']]
df1.head()

In [ ]:
df1_scaled = minmax_scale(df1)
df1_scaled.head()

#### Split Features and Target

In [ ]:
scaled_features = df1_scaled[['area_and_furnishingstatus_and_mainroad_to_bed_and_bath_stories_ratio']] 
scaled_features = scaled_features.values.reshape(-1, 1)
target = df1_scaled.price
print(scaled_features.shape)
print(target.shape)

#### Train and Split the scaled dataframe

In [ ]:
X_train, X_test, y_train, y_test = train_test_splt(scaled_features, target)
Train_test_details(X_train, X_test, y_train, y_test)

#### Re-building the model with Scaled dataframe

In [ ]:
model = LinearRegression()
LinearRegr = training_model(X_train, X_test, y_train, y_test, model)

In [ ]:
train_predictions = predictions_phase(X_train, LinearRegr)
test_predictions = predictions_phase(X_test, LinearRegr)

#### Combined prediction values dataframe

In [ ]:
test_predictions_andy_test, train_predictions_andy_train = prediction_dataframes(train_predictions,y_train,test_predictions, y_test)

#### Error analysis

In [ ]:
print(f'Train data prediction score \n {error_analysis(y_train, train_predictions)}')
print(f'Test data prediction score \n {error_analysis(y_test, test_predictions)}')

#### Plotting predictions from the scaled dataset

In [ ]:
plot_predictions(X_train, y_train, train_predictions, X_test, y_test, test_predictions)

### Conclusion notes on MinMaxScaler and LinearRegression

r2 of Training data : 0.10432734033803603<br>
r2 of Training data : r2 : 0.0681713714203418<br>
The MinMaxScaler also did not result in good accuracy. I did not inverse transform since the prediction were low. This hypothesis is not proved right and the experiment of using MinMaxScaler to acheive better predictions ends.

------

# Using Lasso and Ridge Regression to deal with multi variable and data with multicollinearity

Since we concluded that Linear Regression on field prepared using Variance Inflation Factor process and Linear regression on MinMaxScaler transformed data is not improving model improvement and performance, we are left with multi variable and fields which have multicollinearity.
<BR><BR>
Lasso and Ridge regression works well with multicollinearity data since they reduce the standard error by adding some bias in the estimates of the regression.<br>
In order to further improve the model fit and predictions, we are using GridSearch CV to help identify best hypermeter to use in the model    

# GridSearchCV and Lasso regression
Using GridSearchCV to identify best value for alpha (hyperparameter) for Lasso regression.<br>
Use it to build and fit the model and perform predictions.

In [ ]:
from sklearn import linear_model
from sklearn.model_selection import GridSearchCV, KFold

In [ ]:
# Using all the features as provided in the original dataset
features = housing_df_postcorr[['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad',
       'guestroom', 'basement', 'airconditioning', 'parking', 'prefarea',
       'furnishingstatus']]
# features = features.values.reshape(-1, 1)
target = housing_df_postcorr.price
print(f"Shape of the features: {features.shape}")
print(features.head(5))

print(f"\nShape of the Target: {target.shape}")
print(target.head(5))


In [ ]:
params = {"alpha":np.arange(0.00001, 10, 500)}
kf=KFold(n_splits=5,shuffle=True, random_state=42)

lasso_model = linear_model.Lasso()

# GridSearchCV with model, params and folds.
lasso_gridsearch=GridSearchCV(lasso_model, param_grid=params, cv=kf)
lasso_gridsearch.fit(features, target)

print(f"Best Params for alpha is {lasso_gridsearch.best_params_['alpha']}")
best_alpha = lasso_gridsearch.best_params_['alpha']

In [ ]:
# using best alpha parameter from GridSearch CV
lasso_model = linear_model.Lasso(alpha=best_alpha)

In [ ]:
X_train, X_test, y_train, y_test = train_test_splt(features, target)
lasso_model.fit(X_train, y_train)

In [ ]:
print(f"Trained model score : {lasso_model.score(X_train, y_train)}")
print(f"Adjusted R squared : {1 - (1-lasso_model.score(X_train, y_train))*(len(y_train)-1)/(len(y_train)-X_train.shape[1]-1)}")

In [ ]:
lasso_model_coef = [float(x) for x in np.abs(lasso_model.coef_)]
lasso_model_coef_array = np.asarray(clf_coef)
feature_names = features.columns.tolist()

#### Features and its estimated coefficients based on Lasso Regression

In [ ]:
coefficients_df = pandas.DataFrame()
coefficients_df['features'] = feature_names
coefficients_df['coefficients'] = lasso_model_coef
coefficients_df

In [ ]:
# plotting the Column Names and Importance of Columns. 
plt.bar(feature_names, lasso_model_coef_array)
plt.xticks(rotation=90)
plt.grid()
plt.title("Feature Selection Based on Lasso")
plt.xlabel("Features")
plt.ylabel("Importance")
plt.ylim(0, 1200000)
plt.show()

Feature selection<br>
Selecting features with importance > 200000

In [ ]:
# Subsetting the features which has more than 200000 importance.
feature_subset=np.array(feature_names)[clf_coef>200000]
print("Selected Feature Columns: {}".format(feature_subset))

In [ ]:
nf = features[feature_subset]
nf.head()

#### Building and fitting the model with new set of features list

In [ ]:
X_train, X_test, y_train, y_test = train_test_splt(nf, target)

In [ ]:
lasso_model.fit(X_train, y_train)

In [ ]:
print(f"Trained model score : {lasso_model.score(X_train, y_train)}")
print(f"Adjusted R squared : {1 - (1-lasso_model.score(X_train, y_train))*(len(y_train)-1)/(len(y_train)-X_train.shape[1]-1)}")

#### Notes on feature selection

Tried with multiple thresholds, the feature selection process has reduced the r2 and Adjusted r2 score for any threshold. The goal was to check if trained model accuracy increases due to the feature importance process. Since there is negative effect, we will ignore the feature selection process.

In [ ]:
# Getting the Training and the test data from original set of features
X_train, X_test, y_train, y_test = train_test_splt(features, target)

In [ ]:
# Fitting the model with new training data
lasso_model.fit(X_train, y_train)

In [ ]:
print(f"Trained model score : {lasso_model.score(X_train, y_train)}")
print(f"Adjusted R squared : {1 - (1-lasso_model.score(X_train, y_train))*(len(y_train)-1)/(len(y_train)-X_train.shape[1]-1)}")

#### Predictions using Lasso model

In [ ]:
train_predictions = predictions_phase(X_train, clf)
test_predictions = predictions_phase(X_test, clf)

In [ ]:
test_predictions_andy_test, train_predictions_andy_train = prediction_dataframes(train_predictions,y_train,test_predictions, y_test)

In [ ]:
print(f'Train data prediction score \n {error_analysis(y_train, train_predictions)}')
print(f'Test data prediction score \n {error_analysis(y_test, test_predictions)}')

### Conclusion notes on GridSearch CV and Lasso Regression
r2 of Trainig data : 0.6599646461720708<br>
r2 of Test data : 0.709204704581462<br>

Lasson regression fit and predicted really well with an r2 score of 0.7092.

---

# GridSearchCV and Ridgeregression
Mimic the same process using GridSearch CV to identify best hyperparameters for the Ridge Regression model

In [ ]:
params = {"alpha":np.arange(0.00001, 0.0001, 0.001)}
kf=KFold(n_splits=5,shuffle=True, random_state=42)
Ridge_model = linear_model.Ridge()

# GridSearchCV with model, params and folds.
Ridge_gridsearch=GridSearchCV(Ridge_model, param_grid=params, cv=kf)
Ridge_gridsearch.fit(features, target)
print(f"Best Params for alpha is {Ridge_gridsearch.best_params_['alpha']}")

best_alpha = Ridge_gridsearch.best_params_['alpha']

In [ ]:
# using the best value for alpha based on GridSearch
ridge_model = linear_model.Ridge(alpha=best_alpha)

In [ ]:
ridge_model.fit(X_train, y_train)

#### Features and its estimated coefficients based on Ridge Regression

In [ ]:
coefficients_df = pandas.DataFrame()
coefficients_df['features'] = features.columns.tolist()
coefficients_df['coefficients'] = [float(x) for x in np.abs(ridge_model.coef_)]
coefficients_df

In [ ]:
print(f"Ridge regression Trained model score : {ridge_model.score(X_train, y_train)}")
print(f"Adjusted R squared : {1 - (1-ridge_model.score(X_train, y_train))*(len(y_train)-1)/(len(y_train)-X_train.shape[1]-1)}")

Predictions using Ridge model

In [ ]:
train_predictions = predictions_phase(X_train, ridge_model)
test_predictions = predictions_phase(X_test, ridge_model)

In [ ]:
test_predictions_andy_test, train_predictions_andy_train = prediction_dataframes(train_predictions,y_train,test_predictions, y_test)

In [ ]:
print(f'Train data prediction score \n {error_analysis(y_train, train_predictions)}')
print(f'Test data prediction score \n {error_analysis(y_test, test_predictions)}')

#### Conclusion notes on GridSearch CV and Ridge Regression
r2 of Trainig data : 0.6599646461720694<br>
r2 of Test data : 0.7092047042418796<br>

Ridge regression fit and predicted really well with an r2 score of 0.7092.

-----------

# <b>Conclusion Notes</b>
House Price prediction data is multi variable and suffers with multicollinearity. <BR>
We tried removing multicollinearity using Variance Inflation Factor (VIF) but it has it's own challenges. The problem is identifying the appropriate way to unify multiple fields to capture the importance of multiple fields into one and drop those individual fields. We got a weak prediction using this method <br>
Rescaling using MinMaxScaler on VIF data also did not result in good predictions. <br>
This concluded our attempt to reduce dimension to avoid multicollinearity and fit into LinearRegression.<br>
We used Lasso and Ridge regression since they handle multi variable data well and also deals with multicollinearity really well.<br>
In addition, I used GridSearch CV to identify best hyperparameter to use to build, fit and predict using the model.<br>
This has improved the accuracy, achieved r2 score = 0.7092047042418796 on the test data.
